In [1]:
import os
import requests
import numpy as np
import pandas as pd
from functools import lru_cache
from datetime import datetime, timedelta

# Configuration
EIA_API_KEY = os.getenv('EIA_API_KEY') # Ensure this is set in your environment!
RENEWABLE_CODES = {"SUN", "WND", "WAT", "GEO", "NUC"}
EMISSIONS = {"COL": 2200, "NG": 900, "OIL": 1600, "OTH": 1000} # lbs CO2 per MWh

@lru_cache(maxsize=None)
def fetch_eia_hourly(region: str) -> pd.DataFrame:
    """Fetch recent hourly demand (Real Data)."""
    url = 'https://api.eia.gov/v2/electricity/rto/region-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(days=30) # Last 30 days
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'offset': 0,
        'length': 5000,
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json().get('response', {}).get('data', [])
        df = pd.DataFrame(data)
        if df.empty: return pd.DataFrame()
        df['datetime'] = pd.to_datetime(df['period'])
        df['demand_MW'] = df['value'].astype(float)
        return df.sort_values('datetime')
    except Exception as e:
        print(f"Error fetching demand for {region}: {e}")
        return pd.DataFrame()

@lru_cache(maxsize=None)
def fetch_eia_prices(region: str) -> pd.DataFrame:
    """Fetch wholesale LMP prices (Real Data)."""
    url = "https://api.eia.gov/v2/electricity/wholesale-prices/data/"
    params = {
        "api_key": EIA_API_KEY,
        "frequency": "hourly",
        "facets[respondent][]": region,
        "facets[type][]": "LMP", # Locational Marginal Price
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "length": 5000,
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json().get("response", {}).get("data", [])
        df = pd.DataFrame(data)
        if df.empty: return pd.DataFrame()
        df['datetime'] = pd.to_datetime(df['period'])
        df['price'] = df['value'].astype(float)
        # Average across nodes if multiple exist for the region
        return df.groupby('datetime')['price'].mean().reset_index()
    except Exception as e:
        print(f"Error fetching prices for {region}: {e}")
        return pd.DataFrame()

@lru_cache(maxsize=None)
def fetch_eia_fuelmix(region: str) -> tuple:
    """Fetch fuel mix and return (renewable_pct, co2_intensity)."""
    url = "https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/"
    params = {
        "api_key": EIA_API_KEY,
        "frequency": "hourly",
        "facets[respondent][]": region,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "length": 5000,
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json().get("response", {}).get("data", [])
        if not data: return (np.nan, np.nan)
        
        df = pd.DataFrame(data)
        df['value'] = df['value'].astype(float)
        
        # Calculate totals
        total_gen = df['value'].sum()
        if total_gen == 0: return (np.nan, np.nan)
        
        # Renewable Share
        renew_gen = df[df['fueltype'].isin(RENEWABLE_CODES)]['value'].sum()
        renew_pct = renew_gen / total_gen
        
        # CO2 Intensity (lbs/MWh)
        df['emissions'] = df.apply(lambda x: x['value'] * EMISSIONS.get(x['fueltype'], 1000), axis=1)
        co2_intensity = df['emissions'].sum() / total_gen
        
        return renew_pct, co2_intensity
    except Exception as e:
        print(f"Error fetching fuel mix for {region}: {e}")
        return (np.nan, np.nan)

@lru_cache(maxsize=None)
def fetch_temperature(lat: float, lon: float) -> float:
    """Fetch 60-day historical mean temperature from Open-Meteo."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat,
        'longitude': lon,
        'daily': 'temperature_2m_mean',
        'past_days': 60,
        'timezone': 'UTC',
    }
    try:
        r = requests.get(url, params=params, timeout=5)
        temps = r.json().get('daily', {}).get('temperature_2m_mean', [])
        return np.mean(temps) if temps else np.nan
    except Exception as e:
        print(f"Error fetching temp for {lat},{lon}: {e}")
        return np.nan

In [2]:
import geopandas as gpd
from shapely.geometry import Point

# 1. Define Regions and Coordinates (Centers)
region_coords = {
    'CAL': (36.5, -119.5), 'CAR': (35.5, -80.0), 'CENT': (38.5, -94.5),
    'FLA': (28.0, -82.0), 'MIDA': (39.0, -77.0), 'MIDW': (42.0, -89.0),
    'NE': (42.5, -72.5), 'NY': (42.9, -75.3), 'NW': (45.5, -120.5),
    'SE': (33.0, -84.0), 'SW': (36.0, -111.5), 'TEN': (36.0, -86.0),
    'TEX': (31.0, -99.0),
}

# 2. Map Hexes to Regions (Spatial Join)
# Ensure hex_us_proj exists from previous notebook cells
if 'hex_us_proj' in locals():
    print("Mapping hexes to nearest grid regions...")
    region_gdf = gpd.GeoDataFrame(
        [{"region": r, "geometry": Point(lon, lat)} for r, (lat, lon) in region_coords.items()],
        crs="EPSG:4326"
    ).to_crs(hex_us_proj.crs)
    
    hex_us_proj["region"] = hex_us_proj.geometry.centroid.apply(
        lambda geom: region_gdf.iloc[region_gdf.distance(geom).idxmin()]["region"]
    )

# 3. Fetch Real Data Loop
records = []
print("Fetching real data from EIA and Open-Meteo...")
for region, (lat, lon) in region_coords.items():
    print(f"  Processing {region}...")
    
    # A. Fetch Data
    df_load = fetch_eia_hourly(region)
    df_price = fetch_eia_prices(region)
    renew_raw, co2_raw = fetch_eia_fuelmix(region)
    temp_raw = fetch_temperature(lat, lon)
    
    # B. Compute Metrics
    # Load Stress (Mean Demand) & Stability (Inverse of Volatility)
    if not df_load.empty:
        load_raw = df_load['demand_MW'].mean()
        # Volatility = standard deviation of demand
        volatility_raw = df_load['demand_MW'].std() 
    else:
        load_raw, volatility_raw = np.nan, np.nan

    # Price
    price_raw = df_price['price'].mean() if not df_price.empty else np.nan

    records.append({
        "region": region,
        "price_raw": price_raw,       # Profitability
        "load_raw": load_raw,         # Profitability (Load Stress)
        "renew_raw": renew_raw,       # Sustainability
        "co2_raw": co2_raw,           # Sustainability
        "volatility_raw": volatility_raw, # Sustainability (Stability)
        "temp_raw": temp_raw,         # Sustainability (Cooling)
    })

# 4. Merge Data
df_real = pd.DataFrame(records)
# Fill missing with means to prevent blank maps
df_real = df_real.fillna(df_real.mean(numeric_only=True))

# Merge into Hex Grid
hex_scored = hex_us_proj.merge(df_real, on="region", how="left")

# 5. Normalization (0-1 Scale)
cols = ["price_raw", "load_raw", "renew_raw", "co2_raw", "volatility_raw", "temp_raw"]
for col in cols:
    min_val = hex_scored[col].min()
    max_val = hex_scored[col].max()
    hex_scored[f"n_{col}"] = (hex_scored[col] - min_val) / (max_val - min_val)

# 6. Scoring Logic (60% Sustainability / 40% Profitability)

# Profitability (40%): Low Price is good, Low Load Stress is good
hex_scored["score_profit"] = (
    0.60 * (1 - hex_scored["n_price_raw"]) + 
    0.40 * (1 - hex_scored["n_load_raw"])
)

# Sustainability (60%): High Renew, Low CO2, Low Volatility (High Stability), Low Temp
hex_scored["score_sustain"] = (
    0.30 * hex_scored["n_renew_raw"] + 
    0.30 * (1 - hex_scored["n_co2_raw"]) + 
    0.20 * (1 - hex_scored["n_volatility_raw"]) + 
    0.20 * (1 - hex_scored["n_temp_raw"])
)

# Final GridCast Score
hex_scored["dc_score"] = (
    0.40 * hex_scored["score_profit"] + 
    0.60 * hex_scored["score_sustain"]
)

# 7. Visualization
hex_scored_wgs = hex_scored.to_crs("EPSG:4326")
print("Map generated.")
hex_scored_wgs.explore(
    column="dc_score",
    cmap="viridis",
    legend=True,
    tooltip=["region", "dc_score", "score_sustain", "score_profit", "price_raw", "renew_raw"]
)

Fetching real data from EIA and Open-Meteo...
  Processing CAL...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_4729/1790633309.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


Error fetching prices for CAL: 404 Client Error: Not Found for url: https://api.eia.gov/v2/electricity/wholesale-prices/data/?api_key=HbAcakS9aIGICCg0lr7ovABZuXq8iZ9Gt1JNWpu1&frequency=hourly&facets%5Brespondent%5D%5B%5D=CAL&facets%5Btype%5D%5B%5D=LMP&sort%5B0%5D%5Bcolumn%5D=period&sort%5B0%5D%5Bdirection%5D=desc&length=5000
Error fetching fuel mix for CAL: 'value'
  Processing CAR...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_4729/1790633309.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


Error fetching prices for CAR: 404 Client Error: Not Found for url: https://api.eia.gov/v2/electricity/wholesale-prices/data/?api_key=HbAcakS9aIGICCg0lr7ovABZuXq8iZ9Gt1JNWpu1&frequency=hourly&facets%5Brespondent%5D%5B%5D=CAR&facets%5Btype%5D%5B%5D=LMP&sort%5B0%5D%5Bcolumn%5D=period&sort%5B0%5D%5Bdirection%5D=desc&length=5000
Error fetching fuel mix for CAR: 'value'
  Processing CENT...


...

/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_4729/1790633309.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


Error fetching prices for TEN: 404 Client Error: Not Found for url: https://api.eia.gov/v2/electricity/wholesale-prices/data/?api_key=HbAcakS9aIGICCg0lr7ovABZuXq8iZ9Gt1JNWpu1&frequency=hourly&facets%5Brespondent%5D%5B%5D=TEN&facets%5Btype%5D%5B%5D=LMP&sort%5B0%5D%5Bcolumn%5D=period&sort%5B0%5D%5Bdirection%5D=desc&length=5000
Error fetching fuel mix for TEN: 'value'
  Processing TEX...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_4729/1790633309.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


Error fetching prices for TEX: 404 Client Error: Not Found for url: https://api.eia.gov/v2/electricity/wholesale-prices/data/?api_key=HbAcakS9aIGICCg0lr7ovABZuXq8iZ9Gt1JNWpu1&frequency=hourly&facets%5Brespondent%5D%5B%5D=TEX&facets%5Btype%5D%5B%5D=LMP&sort%5B0%5D%5Bcolumn%5D=period&sort%5B0%5D%5Bdirection%5D=desc&length=5000
Error fetching fuel mix for TEX: 'value'


NameError: name 'hex_us_proj' is not defined